In [1]:
import sys
sys.path.insert(0,'..')
from warnings import filterwarnings
filterwarnings("ignore")
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
%autoreload
import os
import random
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from apex import amp
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from source.version9.data import trainLoader
from source.version9.model import ResNestModel
from source.version9.train import trainModel
from source.version9.loss import BCELoss
from catalyst.data.sampler import BalanceClassSampler

In [3]:
SEED = 42

def seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    return None

seed(42)

In [4]:
def train(fold):
    loader = {}
    loader['image_path'] = '../../data/cdeotte/train/train/'
    loader['label_path'] = '../../data/cdeotte/data.csv'
    loader['fold_idx'] = fold
    train, valid = trainLoader(**loader)
    params = {}
    params['batch_size'] = 15
    params['num_workers'] = 4
    params['drop_last'] = True
    train = DataLoader(train, shuffle=True, **params)
    valid = DataLoader(valid, **params)
    model = ResNestModel()
    model = model.to('cuda:0')
    optimizer = AdamW(model.parameters(), lr=1e-05, weight_decay=0.)
    schedular = ReduceLROnPlateau(optimizer, factor=0.5, patience=0, min_lr=1e-8)
    model, optimizer = amp.initialize(model, optimizer, opt_level='O2', verbosity=False)
    trainer = {}
    trainer['model'] = model
    trainer['train_data'] = train
    trainer['valid_data'] = valid
    trainer['loss_fn'] = BCELoss()
    trainer['optimizer'] = optimizer
    trainer['save_path'] = '../../model/version9/model_{}.pt'.format(fold)
    trainer['epochs'] = 15
    trainer['batch'] = 15
    trainer['scheduler'] = schedular
    trainModel(**trainer)
    model.cpu()
    del model
    return None

In [ ]:
train(0)

Train Images: 35419 Valid Images: 6527


Using cache found in /root/.cache/torch/hub/facebookresearch_semi-supervised-ImageNet1K-models_master
100% 35415/35415 [07:55<00:00, 74.47it/s, trn_ls=0.3035, val_ls=0.1333, val_mt=0.8727]
100% 35415/35415 [07:51<00:00, 75.09it/s, trn_ls=0.2275, val_ls=0.1236, val_mt=0.8763]
100% 35415/35415 [07:57<00:00, 74.22it/s, trn_ls=0.2035, val_ls=0.1236, val_mt=0.8838]
100% 35415/35415 [07:52<00:00, 74.94it/s, trn_ls=0.1795, val_ls=0.1217, val_mt=0.8941]
100% 35415/35415 [07:50<00:00, 75.19it/s, trn_ls=0.1707, val_ls=0.1177, val_mt=0.9033]
100% 35415/35415 [07:51<00:00, 75.16it/s, trn_ls=0.1627, val_ls=0.1144, val_mt=0.9064]
100% 35415/35415 [07:49<00:00, 75.46it/s, trn_ls=0.1534, val_ls=0.1219, val_mt=0.8943]
100% 35415/35415 [07:49<00:00, 75.48it/s, trn_ls=0.1444, val_ls=0.1128, val_mt=0.9115]
100% 35415/35415 [07:49<00:00, 75.42it/s, trn_ls=0.1393, val_ls=0.1119, val_mt=0.9113]
100% 35415/35415 [07:49<00:00, 75.44it/s, trn_ls=0.1345, val_ls=0.1104, val_mt=0.9122]
100% 35415/35415 [07:49<00:0

In [ ]:
train(1)

Train Images: 35397 Valid Images: 6535


Using cache found in /root/.cache/torch/hub/facebookresearch_semi-supervised-ImageNet1K-models_master
100% 35385/35385 [07:47<00:00, 75.61it/s, trn_ls=0.2994, val_ls=0.1518, val_mt=0.8502]
100% 35385/35385 [07:48<00:00, 75.51it/s, trn_ls=0.2249, val_ls=0.1387, val_mt=0.8844]
100% 35385/35385 [07:48<00:00, 75.57it/s, trn_ls=0.2026, val_ls=0.1384, val_mt=0.8908]
100% 35385/35385 [07:48<00:00, 75.46it/s, trn_ls=0.1852, val_ls=0.1299, val_mt=0.9078]
100% 35385/35385 [07:48<00:00, 75.51it/s, trn_ls=0.1721, val_ls=0.1269, val_mt=0.9119]
100% 35385/35385 [07:48<00:00, 75.59it/s, trn_ls=0.1590, val_ls=0.1338, val_mt=0.9084]
 85% 29970/35385 [06:17<01:09, 77.93it/s, trn_ls=0.14430]

In [ ]:
train(2)

In [ ]:
train(3)

In [ ]:
train(4)